# 02 — Fit and tune random forests

Compare the same five feature variants as the original logistic workflow. Each forest is tuned using the **same candidate settings and the same forward event-purged folds**. Selection minimizes row-weighted validation log loss, fixed in advance. ROC-AUC, Brier score, average precision and accuracy are also recorded.

The October 3 outer validation is a diagnostic check, not the tuning target. October 4–5 outcomes never determine settings, features, threshold or importance rankings.

In [ ]:
# Locate this add-on folder whether Jupyter starts here or in 02_notebooks.
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image

here = Path.cwd().resolve()
helper_dir = next((p for base in [here, *here.parents]
                   for p in [base, base / '02_notebooks']
                   if (p / 'rf_common.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('Keep rf_common.py beside these notebooks and start Jupyter in the extracted add-on.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
PACKAGE_ROOT = helper_dir.parent
import rf_common as rf
print('Add-on folder:', PACKAGE_ROOT)

In [ ]:
# Use the exact run created in notebook 01; no original results are overwritten.
OUTPUT_BASE = Path(os.environ.get('WALLET_RF_OUTPUT_BASE', str(PACKAGE_ROOT / '04_results' / 'random_forest'))).expanduser().resolve()
override = os.environ.get('WALLET_RF_RUN_DIR')
RUN_DIR = Path(override).expanduser().resolve() if override else Path(json.loads((OUTPUT_BASE / 'latest_run.json').read_text())['run_dir'])
CONFIG = json.loads((RUN_DIR / 'run_config.json').read_text())
print('Run:', RUN_DIR)
print('Evaluation status:', CONFIG['evaluation_status'])

## 1. Inspect forward cross-validation before fitting

With the supplied week and original split there are three full development days (September 30–October 2). Therefore the default is **two daily forward folds**, not ordinary five-fold shuffled CV:

| Fold | Training entries | Validation entries |
|---|---|---|
| 1 | Before October 1 | October 1 |
| 2 | Before October 2 | October 2 |

Training also requires recorded resolution before the fold starts. Validation outcomes must resolve before the fold ends. Entire validation event groups are purged from training, including earlier members. Folds that are too small are reported; fewer than two usable folds raises an error rather than silently changing methods.

Using a longer period would permit more informative forward folds.

In [ ]:
import pandas as pd
frames = rf.load_frames(RUN_DIR)
all_keys = pd.read_parquet(RUN_DIR / 'all_entry_keys.parquet')
folds, fold_audit = rf.forward_event_folds(frames['train'], all_keys, CONFIG)
display(fold_audit)
print('Usable forward folds:', len(folds))

## 2. Tune, check validation and refit

For every candidate/fold, a new median imputer is fitted **only on that fold's training rows**. No scaling is needed for the forest. The logistic comparator uses the original fixed `C=1`, median imputation and training-fitted scaling, with missing indicators left unscaled.

For each variant the code:

1. Chooses RF settings using inner forward CV only.
2. Fits a development forest and logistic reference, then evaluates October 3.
3. Calculates grouped permutation importance on that outer validation set.
4. Refits both models on the original final-refit rows before October 4 and saves them.
5. Exports absolute standardized logistic coefficients and individual RF Gini scores from those saved final models.

The candidate grid varies maximum depth, minimum leaf size and feature subsampling. The number of trees, class weights (`None`) and 0.5 classification threshold stay fixed. There is no feature deletion based on p-values.

Runtime depends on the real cohort and machine. Defaults require 80 CV forest fits (5 variants × 8 settings × 2 folds), plus development/final fits and permutation scoring. Reduce resources in notebook 01 before starting a new run if needed.

In [ ]:
validation_metrics, selected_parameters = rf.fit_models(RUN_DIR, CONFIG)
display(validation_metrics[['model', 'variant', 'rows', 'roc_auc', 'log_loss',
                            'brier', 'pr_auc_average_precision', 'accuracy_threshold_0_5']])
display(pd.DataFrame(selected_parameters).T)

## 3. Review individual feature importance

The main comparison now uses **absolute standardized coefficients** for logistic regression and **Gini impurity importance** for random forest. Price, price² and price³ appear as separate columns. All five feature variants are exported.

Logistic continuous predictors were standardized using the refit cohort's fitted scaler. The signed coefficients are retained. Missingness indicators remain unscaled, so they are listed separately instead of mixed into the continuous-feature rankings. Coefficient magnitude is not a p-value; the models use L2 regularization with C=1.

RF Gini scores describe each column's contribution to training impurity reduction and sum to one across all columns. They can favor features with many possible split points. The two algorithms' score magnitudes are not directly comparable.

**Price terms are correlated:** their individual coefficients/importances should not be read as independent signals or isolated effects. The existing grouped validation permutation scores remain a supporting check, including the `ALL_HISTORY_TOGETHER` row. Permutation shuffle SD is not a confidence interval or p-value. These are not XGBoost gain scores.

In [ ]:
# Tables are automatically generated by fit_models; no extra model fitting occurs.
importance = pd.read_csv(RUN_DIR / 'variable_importance_all_models.csv')
for variant in ['baseline', 'fixed24h_5']:
    print('\n', variant)
    for kind in ['logistic', 'random_forest']:
        table = importance.loc[importance.variant.eq(variant) & importance.model.eq(kind)]
        print(kind, ': continuous-predictor ranking')
        display(table.loc[~table.is_missing_indicator,
                          ['label', 'coefficient', 'importance', 'measure', 'rank_among_nonmissing_features']]
                .sort_values('importance', ascending=False))
        print('Missing indicators (unscaled in logistic):')
        display(table.loc[table.is_missing_indicator, ['label', 'coefficient', 'importance', 'measure']])
print('Supporting grouped validation permutation scores remain in validation_grouped_permutation_importance.csv.')
print('Next: 03_rf_validation.ipynb. To update only importance for existing models, use notebook 04.')

## Method references

- [scikit-learn: RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [scikit-learn: time-dependent cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-of-time-series-data)
- [scikit-learn: permutation importance and correlated features](https://scikit-learn.org/stable/modules/permutation_importance.html)

The custom split adds event purging and recorded-resolution maturity rules to chronological validation; ordinary `TimeSeriesSplit` alone would not enforce these project-specific checks.